# 01 — Data Loading

Download the Lending Club dataset (2007–2020 Q3) from Kaggle via `kagglehub`, locate the gzip file, and load a 31-column subset into a pandas DataFrame.

**Dataset:** [Lending Club 2007–2020 Q3](https://www.kaggle.com/datasets/ethon0426/lending-club-20072020q1)  
**Rows:** ~2.9 million  
**Format:** gzip-compressed CSV

In [ ]:
import kagglehub
import os
import zipfile

# ── Download latest version from Kaggle ──────────────────────────
path = kagglehub.dataset_download("ethon0426/lending-club-20072020q1")
print("Path to dataset files:", path)

# ── Locate the gzip / csv file ────────────────────────────────────
data_file = None
for item in os.listdir(path):
    if item.endswith((".gzip", ".csv")):
        data_file = os.path.join(path, item)
        break

if data_file:
    file_path = data_file
    print("Data file found:", file_path)
else:
    # Fallback: unzip if needed
    unzip_dir = "data"
    os.makedirs(unzip_dir, exist_ok=True)
    for item in os.listdir(path):
        if item.endswith(".zip"):
            zip_path = os.path.join(path, item)
            with zipfile.ZipFile(zip_path, "r") as zip_ref:
                zip_ref.extractall(unzip_dir)
            print(f"Unzipped {item} to {unzip_dir}")
    for root, dirs, files in os.walk(unzip_dir):
        for file in files:
            if file.endswith((".gzip", ".csv")):
                data_file = os.path.join(root, file)
                break
        if data_file:
            break
    if data_file:
        file_path = data_file
        print("Data file found:", file_path)
    else:
        raise FileNotFoundError("Data file not found. Check the downloaded contents.")    

In [ ]:
import pandas as pd
import gzip

# ── Columns selected for analysis ─────────────────────────────────
# 31 columns covering loan terms, borrower profile, credit history
final_cols = [
    'loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade',
    'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d',
    'loan_status', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line',
    'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
    'initial_list_status', 'collections_12_mths_ex_med', 'application_type', 'acc_now_delinq',
    'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim'
]

# ── Read in chunks (file is ~2.9M rows — too large to load at once) ─
chunk_size = 100_000
try:
    chunks = pd.read_csv(
        file_path,
        usecols=lambda c: c in final_cols,
        compression='gzip',
        chunksize=chunk_size,
        low_memory=False
    )
except (gzip.BadGzipFile, pd.errors.ParserError):
    chunks = pd.read_csv(
        file_path,
        usecols=lambda c: c in final_cols,
        chunksize=chunk_size,
        low_memory=False
    )

df = pd.concat(chunks, ignore_index=True)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()    

In [ ]:
# ── Basic data overview ────────────────────────────────────────────
print("Shape:", df.shape)
print("\nColumn dtypes:")
print(df.dtypes)
print("\nMissing values (top 10):")
print(df.isnull().sum().sort_values(ascending=False).head(10))
print("\nTarget variable distribution:")
print(df['loan_status'].value_counts())    

In [ ]:
import os

# ── Save a 100-row sample to data/raw/ for GitHub visibility ──────
# (Full dataset ~600MB — too large for Git)
raw_dir = '../data/raw'
os.makedirs(raw_dir, exist_ok=True)
df.head(100).to_csv(f'{raw_dir}/sample_100rows.csv', index=False)
print("Saved 100-row sample to data/raw/sample_100rows.csv")
print("Full dataset must be downloaded via kagglehub (see data/raw/README.md)")    